<a href="https://colab.research.google.com/github/Melissa-Etes/Sprint2_grupo54/blob/main/EC_Sprint2_HospiDataSUS_Clusterizacao_Grupo54.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clusterização de Municípios — HospiData SUS - GRUPO 54
Análise de padrões de sobrecarga hospitalar por município, usando K-Means.

## 1. Importações e carga dos dados

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

SEED = 1224
np.random.seed(SEED)

##### Carregando Dados do GitHub

In [2]:
# Carrega os dados tratados (exportados do Oracle via ETL) direto do GitHub
dados = pd.read_csv('https://raw.githubusercontent.com/Melissa-Etes/Sprint2_grupo54/main/data/dados_tratados_datasus.csv')
dados.head()

,COD_MUNICIPIO,MUNICIPIO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,VALOR_MEDIO_INTERNACAO,QTD_ESTAB_HOSPITALARES,POPULACAO,INTERNACOES_POR_MIL_HAB
0,351390,Divinolândia,266,17.95,1.50,1617.77,1,11158,23.84
1,352450,Jaci,180,24.41,1.11,2086.81,1,7613,23.64
2,352750,Lucianópolis,29,3.83,0.00,1144.26,0,2372,12.23
3,351080,Casa Branca,340,12.51,2.94,986.24,1,28083,12.11
4,351940,Ibirá,139,5.41,6.47,2472.68,1,11690,11.89


##### Conferir os dados

In [3]:
dados.shape
dados.isna().sum()

,0
COD_MUNICIPIO,0
MUNICIPIO,0
QTD_INTERNACOES,0
PERMANENCIA_MEDIA,0
TAXA_MORTALIDADE_PCT,0
VALOR_MEDIO_INTERNACAO,0
QTD_ESTAB_HOSPITALARES,0
POPULACAO,0
INTERNACOES_POR_MIL_HAB,0


In [4]:
dados = dados.dropna(subset=['POPULACAO'])
dados['QTD_ESTAB_HOSPITALARES'] = dados['QTD_ESTAB_HOSPITALARES'].fillna(0)

## 2. Correção de viés: Razão Observado/Esperado

A taxa bruta de internações por mil habitantes distorce municípios pequenos (qualquer número de internações vira uma taxa desproporcional). Corrigimos calculando quantas internações seriam **esperadas** para cada município, dado o tamanho da população, e comparando com o valor **observado**.

In [5]:
# Taxa média geral de internação (ponderada pela população total)
taxa_media_geral = dados['QTD_INTERNACOES'].sum() / dados['POPULACAO'].sum() * 1000

# Número esperado de internações para cada município
dados['internacoes_esperadas'] = taxa_media_geral * dados['POPULACAO'] / 1000

# Razão Observado/Esperado: >1 = acima da média (proporcional à população), <1 = abaixo
dados['razao_obs_esperado'] = dados['QTD_INTERNACOES'] / dados['internacoes_esperadas']

dados[['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'internacoes_esperadas', 'razao_obs_esperado']] \
    .sort_values('razao_obs_esperado', ascending=False).head(10)

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,internacoes_esperadas,razao_obs_esperado
0,Divinolândia,11158,266,49.609130,5.361916
1,Jaci,7613,180,33.847850,5.317915
2,Lucianópolis,2372,29,10.546053,2.749844
3,Casa Branca,28083,340,124.858685,2.723079
4,Ibirá,11690,139,51.974434,2.674392
5,Cajuru,23830,282,105.949594,2.661643
6,Vitória Brasil,1794,21,7.976230,2.632823
7,Nantes,2660,31,11.826518,2.621228
8,Emilianópolis,3014,35,13.400423,2.611858
9,Uru,1387,16,6.166684,2.594587


In [6]:
# Conferindo os municípios que antes apareciam como outliers isolados
dados[dados['MUNICIPIO'].isin(['Jaci', 'Divinolândia'])][
    ['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'internacoes_esperadas', 'razao_obs_esperado']
]

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,internacoes_esperadas,razao_obs_esperado
0,Divinolândia,11158,266,49.60913,5.361916
1,Jaci,7613,180,33.84785,5.317915


## 3. Seleção de features

**Importante:** não incluímos `QTD_INTERNACOES` (número bruto) nem `INTERNACOES_POR_MIL_HAB` — ambas ainda carregam o viés de tamanho populacional. Usamos `razao_obs_esperado` no lugar, que já é a versão normalizada dessa informação.

In [7]:
features = dados[['PERMANENCIA_MEDIA', 'TAXA_MORTALIDADE_PCT',
                   'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']]
features.head()

,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
0,17.95,1.50,1,5.361916
1,24.41,1.11,1,5.317915
2,3.83,0.00,0,2.749844
3,12.51,2.94,1,2.723079
4,5.41,6.47,1,2.674392


## 4. Pipeline de padronização + PCA

In [8]:
pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('PCA', PCA(n_components=2, random_state=SEED))
])

embedding_pca = pca_pipeline.fit_transform(features)
projection = pd.DataFrame(columns=['x', 'y'], data=embedding_pca)
projection.head()

,x,y
0,10.380407,2.371760
1,12.194281,4.308478
2,2.449628,-1.664561
3,4.690190,1.411027
4,2.334113,-0.315731


In [9]:
print("Variância explicada (2D):", pca_pipeline.named_steps['PCA'].explained_variance_ratio_.sum())

Variância explicada (2D): 0.5733050188583304


## 5. Método do Cotovelo — escolha do número de clusters

In [10]:
inercia = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(embedding_pca)
    inercia.append(km.inertia_)

fig_cotovelo = px.line(
    x=list(range(1, 10)), y=inercia, markers=True,
    title='Método do Cotovelo',
    labels={'x': 'Número de clusters (k)', 'y': 'Inércia'}
)
fig_cotovelo.show()

In [11]:
# Salva o gráfico como imagem para usar no PPT/README
# (se der erro de kaleido, rode: !pip install kaleido==0.2.1  e reinicie a sessão)
fig_cotovelo.write_image("metodo_cotovelo.png")

## 6. Aplicação do K-Means

In [12]:
K_ESCOLHIDO = 3  # ajuste conforme o gráfico do cotovelo

kmeans = KMeans(n_clusters=K_ESCOLHIDO, random_state=SEED, n_init=10)
projection['cluster'] = kmeans.fit_predict(embedding_pca).astype(str)
projection['municipio'] = dados['MUNICIPIO'].values
dados['cluster'] = projection['cluster'].values
projection.head()

,x,y,cluster,municipio
0,10.380407,2.371760,2,Divinolândia
1,12.194281,4.308478,2,Jaci
2,2.449628,-1.664561,2,Lucianópolis
3,4.690190,1.411027,2,Casa Branca
4,2.334113,-0.315731,2,Ibirá


## 7. Visualização dos clusters

In [13]:
fig_clusters = px.scatter(
    projection, x='x', y='y', color='cluster',
    hover_data=['municipio'],
    title='Clusters de Municípios (Métrica Corrigida - Razão Observado/Esperado)'
)
fig_clusters.update_traces(marker=dict(size=8, opacity=0.9, line=dict(width=0.5, color='#121212')))
fig_clusters.show()

In [14]:
fig_clusters.write_image("clusters_municipios.png")

## 8. Interpretação dos clusters

In [15]:
dados.groupby('cluster')[
    ['PERMANENCIA_MEDIA', 'TAXA_MORTALIDADE_PCT', 'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']
].mean()

,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
cluster,,,,
0,4.351575,5.935500,0.825000,1.165633
1,6.070987,12.223947,4.138158,0.998244
2,6.746022,4.685806,0.548387,1.960972


In [16]:
# Quantos municípios em cada cluster
dados['cluster'].value_counts()

,count
cluster,
0,400
1,152
2,93


## 9. Investigação de outliers

Se algum cluster tiver poucos municípios muito isolados, investigamos individualmente.

In [17]:
dados[dados['cluster'] == '1'][
    ['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'PERMANENCIA_MEDIA',
     'TAXA_MORTALIDADE_PCT', 'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']
]

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
32,Adolfo,4351,41,5.49,17.07,0,2.119432
55,Paranapuã,4031,33,4.67,15.15,1,1.841306
99,Salesópolis,15202,110,5.68,11.82,0,1.627484
100,Marabá Paulista,4573,33,6.15,15.15,0,1.623071
120,Alvinlândia,2885,20,5.35,15.00,0,1.559225
...,...,...,...,...,...,...,...
634,Bom Sucesso de Itararé,3555,8,5.00,25.00,0,0.506145
635,Riversul,5599,12,9.42,8.33,0,0.482054
637,Lupércio,3981,8,6.00,25.00,0,0.451983
642,Taquarivaí,6876,10,4.40,20.00,0,0.327106


# **Confirma outliers como sinal real de vulnerabilidade**

In [18]:
dados[dados['cluster'] == '2'][
    ['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'PERMANENCIA_MEDIA',
     'TAXA_MORTALIDADE_PCT', 'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']
]

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
0,Divinolândia,11158,266,17.95,1.50,1,5.361916
1,Jaci,7613,180,24.41,1.11,1,5.317915
2,Lucianópolis,2372,29,3.83,0.00,0,2.749844
3,Casa Branca,28083,340,12.51,2.94,1,2.723079
4,Ibirá,11690,139,5.41,6.47,1,2.674392
...,...,...,...,...,...,...,...
366,Nova Canaã Paulista,2032,10,9.40,0.00,0,1.106881
415,Itu,168240,776,11.02,6.44,3,1.037426
422,Catiguá,7003,32,10.84,0.00,0,1.027757
501,Mogi Mirim,92558,370,9.06,3.78,2,0.899109


Exportando como csv para levar para o Oracle

In [19]:
dados.to_csv('internacoes_com_clusters.csv', index=False)